# Portfolio: reasoning model mini bằng GRPO (TRL) trên GSM8K

Bạn train `Qwen2.5-0.5B-Instruct` bằng `GRPOTrainer` của TRL với reward kiểm được, rồi chứng minh bằng số rằng nó giỏi
hơn (hoặc không): độ chính xác trước/sau trên 500 câu test ghim có khoảng tin cậy, ít nhất hai seed, đường cong reward và
độ dài, và phân tích lỗi (reward hacking, ăn điểm định dạng). Viết kết quả vào `REPORT.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell `explore` là khung gợi ý: sửa thoải mái, đó là thí nghiệm của bạn.
- Chạy bằng kernel `ai-gpu`. Cấu hình `LAPTOP` (GPU 4GB) và `PC` (GPU 12GB) ở cell cấu hình.
- Chấm: `uv run pytest tracks/ai-engineer/modules/15-reasoning-models/exercises/advanced/01-grpo-reasoning-mini -q`

In [ ]:
# explore: tìm gốc repo, tải GSM8K nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

GSM = ensure("gsm8k")
CACHE = GSM / "cache"
CACHE.mkdir(exist_ok=True)

In [ ]:
import re
from fractions import Fraction

import numpy as np

## 1. Dữ liệu và reward theo giao diện của TRL

Dán `normalize_number`, `extract_answer`, `format_reward`, `correctness_reward` (cùng regex, hàm phụ) từ Basic 01 và
`paired_bootstrap` từ Intermediate 01. Rồi viết:

- `build_rows(questions, answers) -> list[dict]`: mỗi phần tử `{"prompt": [system, user], "answer": đáp số chuẩn hoá}`,
  system là `SYSTEM_COT`, `answers` là cột `answer` thô của GSM8K. TRL truyền mọi cột khác `prompt` vào reward function
  dưới dạng keyword.
- `reward_correct(completions, answer, **kwargs) -> list[float]` và `reward_format(completions, **kwargs) -> list[float]`:
  với dữ liệu hội thoại, mỗi phần tử của `completions` là một list message, nội dung ở `completion[0]["content"]`. Hai hàm
  chỉ gọi lại `correctness_reward` / `format_reward`; trọng số 1.0 và 0.2 đặt ở `GRPOConfig.reward_weights`.

In [ ]:
SYSTEM_COT = "Please reason step by step, and put your final answer within \\boxed{}."


def normalize_number(text: str) -> str | None:
    """Dán từ bài Basic 01 (cùng mọi regex và hàm phụ nó cần)."""
    # TODO
    raise NotImplementedError


def extract_answer(completion: str) -> str | None:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def format_reward(completion: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def correctness_reward(completion: str, gold: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def paired_bootstrap(correct_a, correct_b, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """Dán từ bài Intermediate 01."""
    # TODO
    raise NotImplementedError


def build_rows(questions: list[str], answers: list[str]) -> list[dict]:
    """[{"prompt": [system, user], "answer": đáp số chuẩn hoá}] cho GRPOTrainer."""
    # TODO
    raise NotImplementedError


def reward_correct(completions: list[list[dict]], answer: list[str], **kwargs) -> list[float]:
    """correctness_reward cho từng completion hội thoại."""
    # TODO
    raise NotImplementedError


def reward_format(completions: list[list[dict]], **kwargs) -> list[float]:
    """format_reward cho từng completion hội thoại."""
    # TODO
    raise NotImplementedError

## 2. Cấu hình

Hai cấu hình đã chạy được trên bản tham chiếu (số đo ở `SOLUTION.md`). `PC` là cấu hình của bảng kết quả; `LAPTOP` giảm
batch và độ dài để vừa GPU 4GB, chậm hơn và ít bước hơn: ghi rõ cấu hình bạn dùng trong báo cáo.

In [ ]:
# explore: cấu hình
import torch

PC = dict(steps=60, per_device_train_batch_size=4, gradient_accumulation_steps=16, num_generations=8,
          max_completion_length=384, learning_rate=4e-5, beta=0.0, temperature=0.7, n_eval=500)
LAPTOP = dict(steps=60, per_device_train_batch_size=2, gradient_accumulation_steps=32, num_generations=8,
              max_completion_length=320, learning_rate=4e-5, beta=0.0, temperature=0.7, n_eval=500)
free_gib = torch.cuda.mem_get_info()[0] / 2**30 if torch.cuda.is_available() else 0
CFG = PC if free_gib > 8 else LAPTOP
SEEDS = [0, 1]
print(f"GPU trống {free_gib:.1f} GiB ->", "PC" if CFG is PC else "LAPTOP", CFG)

## 3. Đo model gốc, train, đo lại

Khung dưới đây là một cách làm; đổi nếu bạn có lý do (ghi lý do vào báo cáo). Lưu mọi thứ (log của trainer, câu trả lời
trước/sau) ra `CACHE` để phân tích lại mà không train lại. **Không commit adapter hay checkpoint.**

In [ ]:
# explore: TODO: hàm evaluate(model) greedy trên CFG["n_eval"] câu test đầu: accuracy (bộ trích dễ dãi),
# boxed_correct (như reward), format, độ dài, số câu bị cắt; lưu danh sách đúng/sai từng câu để bootstrap theo cặp.

In [ ]:
# explore: TODO: với mỗi seed: GRPOConfig(... reward_weights=[1.0, 0.2], logging_steps=1, save_strategy="no",
# report_to="none"), GRPOTrainer(model=..., reward_funcs=[reward_correct, reward_format], train_dataset=...,
# peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")),
# trainer.train(), evaluate, lưu trainer.state.log_history.

## 4. Phân tích

- Bảng trước/sau theo seed, hiệu so với model gốc kèm khoảng tin cậy theo cặp; hai cột `accuracy` và `boxed_correct`.
- Đường cong theo bước từ `log_history`: `reward`, `rewards/reward_correct/mean`, `rewards/reward_format/mean`,
  `completions/mean_length`, `frac_reward_zero_std` (tên khoá đúng như TRL 1.17 ghi; in `log_history[-1].keys()` để xem).
- Độ phủ: lấy mẫu 8 lời giải cho 200 câu đầu trước và sau, so pass@1 và pass@8 (mục 3 lý thuyết, Yue và cộng sự 2025).
- Phân tích lỗi: đọc ít nhất 20 câu trả lời sau train, đặc biệt các câu đúng -> sai, và các câu `accuracy` đúng nhưng
  `boxed_correct` sai (hoặc ngược lại).

In [ ]:
# explore: TODO: bảng, đường cong, độ phủ, phân tích lỗi